## Part A: Handling Missing Value

**Import Libraries**

In [20]:
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Simple Imputer
from sklearn.impute import SimpleImputer

# KNN Imputer
from sklearn.impute import KNNImputer

# MICE
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

# For Random Imputation
import random

# For Z score 
from scipy.stats import zscore

# For winsorize
from scipy.stats.mstats import winsorize


**Load Dataset**

In [7]:
# Load Dataset
df = pd.read_csv("patient_data.csv")

# Display First 5 Row
print(df.head())


  patient_id   age  gender region   bmi  blood_pressure  cholesterol  glucose  \
0       P001  25.0    Male  North  22.5             120        180.0     95.0   
1       P002  34.0  Female  South  27.8             130        220.0    110.0   
2       P003  45.0    Male   East   NaN             145        250.0    140.0   
3       P004  29.0  Female   West  24.1             118        190.0     98.0   
4       P005   NaN    Male  North  31.2             150        270.0    160.0   

   disease_risk  
0             0  
1             1  
2             1  
3             0  
4             1  


**Q1 Identify Missing Values and Provide Summary Report**

In [8]:
# Dataset Shape 
print("==================================== Dataset Shape =======================================")
print(df.shape)

# Missing Value in each column
print("===================================== Missing Value In Each Column ========================================")
print(df.isnull().sum())

# Percentage of missing value
missing_per = (df.isnull().sum()) / len(df) * 100

# Summary report
print("===================================== Summary Report ===================================================")
summary = pd.DataFrame({
    "Missing Value" : df.isnull().sum(),
    "Percentage (%)": missing_per.round(2)
})
print(summary)

# Total Missing Value
print("====================================== Total Missing Value ==========================================")
print(df.isnull().sum().sum())

==================================== Dataset Shape =======================================
(200, 9)
===================================== Missing Value In Each Column ========================================
patient_id         0
age               10
gender             6
region            11
bmi               12
blood_pressure     0
cholesterol        5
glucose            3
disease_risk       0
dtype: int64
===================================== Summary Report ===================================================
                Missing Value  Percentage (%)
patient_id                  0             0.0
age                        10             5.0
gender                      6             3.0
region                     11             5.5
bmi                        12             6.0
blood_pressure              0             0.0
cholesterol                 5             2.5
glucose                     3             1.5
disease_risk                0             0.0
=========================

**Q2(a): Simple Imputer (Mean) for BMI**

In [10]:
# Create a copy of dataset
df_mean = df.copy()

# Mean Imputer
mean_imputer  = SimpleImputer(strategy="mean")

# Fill missing BMI Values
df_mean["bmi"] = mean_imputer.fit_transform(df_mean[["bmi"]])

print("Missing BMI Values After Mean Imputation:")
print(df_mean["bmi"].isnull().sum())

print("\nFirst 10 BMI Values")
print(df_mean["bmi"].head(10))

Missing BMI Values After Mean Imputation:
0

First 10 BMI Values
0    22.500000
1    27.800000
2    29.504787
3    24.100000
4    31.200000
5    26.500000
6    35.400000
7    23.800000
8    29.504787
9    28.600000
Name: bmi, dtype: float64


**Q2(b): Simple Imputer (Median) for BMI**                                                                                

In [11]:
# Create a copy of dataset
df_median = df.copy()

# Median Imputer
median_imputer = SimpleImputer(strategy="median")

# Fill missing BMI values
df_median["bmi"] = median_imputer.fit_transform(df_median[["bmi"]])

print("Missing BMI Values After Median Imputation:")
print(df_median["bmi"].isnull().sum())

print("\nFirst 10 BMI Values")
print(df_median["bmi"].head(10))

Missing BMI Values After Median Imputation:
0

First 10 BMI Values
0    22.50
1    27.80
2    28.25
3    24.10
4    31.20
5    26.50
6    35.40
7    23.80
8    28.25
9    28.60
Name: bmi, dtype: float64


**Q2(c): Most Frequent Imputation for Region**

In [12]:
# Create a copy
df_region = df.copy()

# Most Frequent Imputer
region_imputer = SimpleImputer(strategy="most_frequent")

# Fill missing Region values
df_region["region"] = region_imputer.fit_transform(df_region[["region"]]).ravel()

print("Missing Region Values After Imputation:")
print(df_region["region"].isnull().sum())

print("\nFirst 10 Region Values")
print(df_region["region"].head(10))

Missing Region Values After Imputation:
0

First 10 Region Values
0    North
1    South
2     East
3     West
4    North
5    South
6    North
7     East
8     West
9    North
Name: region, dtype: str


**Q2(d): Most Frequent Imputation for Gender**

In [13]:

# Create a copy
df_gender = df.copy()

# Most Frequent Imputer
gender_imputer = SimpleImputer(strategy="most_frequent")

# Fill missing Gender values
df_gender["gender"] = gender_imputer.fit_transform(df_gender[["gender"]]).ravel()

print("Missing Gender Values After Imputation:")
print(df_gender["gender"].isnull().sum())

print("\nFirst 10 Gender Values")
print(df_gender["gender"].head(10))

Missing Gender Values After Imputation:
0

First 10 Gender Values
0      Male
1    Female
2      Male
3    Female
4      Male
5    Female
6    Female
7      Male
8    Female
9      Male
Name: gender, dtype: str


**Q2(e): Compare Results**

In [14]:
comparison = pd.DataFrame({
    "Original": df.isnull().sum(),
    "Mean (BMI)": df_mean.isnull().sum(),
    "Median (BMI)": df_median.isnull().sum(),
    "Region Imputed": df_region.isnull().sum(),
    "Gender Imputed": df_gender.isnull().sum()
})

print(comparison)

                Original  Mean (BMI)  Median (BMI)  Region Imputed  \
patient_id             0           0             0               0   
age                   10          10            10              10   
gender                 6           6             6               6   
region                11          11            11               0   
bmi                   12           0             0              12   
blood_pressure         0           0             0               0   
cholesterol            5           5             5               5   
glucose                3           3             3               3   
disease_risk           0           0             0               0   

                Gender Imputed  
patient_id                   0  
age                         10  
gender                       0  
region                      11  
bmi                         12  
blood_pressure               0  
cholesterol                  5  
glucose                      3  
di

**Part B: Handling Outliers**

**Q3(a). Z-Score Method (Cholesterol & Glucose)**

In [17]:
# Copy Dataset
df_zscore = df.copy()

# Mean and Standard Deviation
chol_mean = df_zscore["cholesterol"].mean()
chol_std = df_zscore["cholesterol"].std()

glu_mean = df_zscore["glucose"].mean()
glu_std = df_zscore["glucose"].std()

# Manual Z-score Formula
df_zscore["chol_zscore"] = (df_zscore["cholesterol"] - chol_mean) / chol_std

df_zscore["glucose_zscore"] = (df_zscore["glucose"] - glu_mean) / glu_std

# Detect Outliers
outliers = df_zscore[
    (abs(df_zscore["chol_zscore"]) > 3) |
    (abs(df_zscore["glucose_zscore"]) > 3)
]

print("Outliers Found")
print(outliers[["patient_id","cholesterol","glucose","chol_zscore","glucose_zscore"]])

# Remove Outliers
df_zscore = df_zscore[
    (abs(df_zscore["chol_zscore"]) <= 3) &
    (abs(df_zscore["glucose_zscore"]) <= 3)
]

print("\nDataset Shape After Removing Outliers:")
print(df_zscore.shape)

Outliers Found
Empty DataFrame
Columns: [patient_id, cholesterol, glucose, chol_zscore, glucose_zscore]
Index: []

Dataset Shape After Removing Outliers:
(192, 11)


**Q3(b). IQR Method (BMI)**

In [18]:

df_iqr = df.copy()

Q1 = df_iqr["bmi"].quantile(0.25)
Q3 = df_iqr["bmi"].quantile(0.75)

IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("Lower Limit :", lower)
print("Upper Limit :", upper)

# Detect Outliers
outliers = df_iqr[
    (df_iqr["bmi"] < lower) |
    (df_iqr["bmi"] > upper)
]

print("\nNumber of BMI Outliers :", len(outliers))

# Remove Outliers
df_iqr = df_iqr[
    (df_iqr["bmi"] >= lower) &
    (df_iqr["bmi"] <= upper)
]

print("Dataset Shape :", df_iqr.shape)

Lower Limit : 9.337499999999993
Upper Limit : 49.23750000000001

Number of BMI Outliers : 0
Dataset Shape : (188, 9)


**Q3(c). Percentile Method**

In [19]:
df_percentile = df.copy()

lower = df_percentile["bmi"].quantile(0.01)
upper = df_percentile["bmi"].quantile(0.99)

# Cap values
df_percentile["bmi"] = df_percentile["bmi"].clip(lower, upper)

print(df_percentile["bmi"].describe())


count    188.000000
mean      29.502936
std        6.304306
min       20.687000
25%       24.300000
50%       28.250000
75%       34.275000
max       43.839000
Name: bmi, dtype: float64


**Q4. Winsorization**

In [21]:
df_win = df.copy()

df_win["bmi"] = winsorize(df_win["bmi"], limits=[0.01,0.01])

print(df_win["bmi"].describe())

count    188.000000
mean      29.506383
std        6.311755
min       20.700000
25%       24.300000
50%       28.250000
75%       34.275000
max       44.200000
Name: bmi, dtype: float64


c:\Users\Dell\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


**Q5. Compare Before vs After**

In [24]:
print("Original Dataset Shape")
print(df.shape)

print("\nAfter Z-score")
print(df_zscore.shape)

print("\nAfter IQR")
print(df_iqr.shape)

print("\nAfter Winsorization")
print(df_win.shape)

print("\nOriginal Summary")
print(df.describe())

print("\nAfter Winsorization Summary")
print(df_win.describe())

Original Dataset Shape
(200, 9)

After Z-score
(192, 11)

After IQR
(188, 9)

After Winsorization
(200, 9)

Original Summary
              age         bmi  blood_pressure  cholesterol     glucose  \
count  190.000000  188.000000      200.000000   195.000000  197.000000   
mean    42.247368   29.504787      147.390000   251.225641  147.208122   
std     10.737211    6.314014       29.275127    64.246468   57.267118   
min     23.000000   20.500000      110.000000   165.000000   82.000000   
25%     33.000000   24.300000      123.750000   194.000000  102.000000   
50%     42.000000   28.250000      138.500000   238.000000  125.000000   
75%     51.000000   34.275000      170.250000   306.500000  180.000000   
max     62.000000   44.200000      220.000000   385.000000  310.000000   

       disease_risk  
count    200.000000  
mean       0.520000  
std        0.500854  
min        0.000000  
25%        0.000000  
50%        1.000000  
75%        1.000000  
max        1.000000  

After Win

c:\Users\Dell\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\numpy\lib\_function_base_impl.py:4786: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  arr.partition(


## Part C: Final Clean Dataset

**Q6. Final Clean Dataset**

In [25]:
df_clean = df.copy()

# Fill Missing Values

df_clean["bmi"] = df_clean["bmi"].fillna(df_clean["bmi"].median())

df_clean["region"] = df_clean["region"].fillna(df_clean["region"].mode()[0])

df_clean["gender"] = df_clean["gender"].fillna(df_clean["gender"].mode()[0])

df_clean["age"] = df_clean["age"].fillna(df_clean["age"].mean())

df_clean["cholesterol"] = df_clean["cholesterol"].fillna(df_clean["cholesterol"].mean())

df_clean["glucose"] = df_clean["glucose"].fillna(df_clean["glucose"].mean())

# Winsorization on BMI
from scipy.stats.mstats import winsorize

df_clean["bmi"] = winsorize(df_clean["bmi"], limits=[0.01,0.01])

print(df_clean.head())

print("\nMissing Values")
print(df_clean.isnull().sum())

print("\nDataset Shape")
print(df_clean.shape)

# Save Dataset
df_clean.to_csv("Final_Clean_Dataset.csv", index=False)

print("\nFinal Dataset Saved Successfully!")

  patient_id        age  gender region    bmi  blood_pressure  cholesterol  \
0       P001  25.000000    Male  North  22.50             120        180.0   
1       P002  34.000000  Female  South  27.80             130        220.0   
2       P003  45.000000    Male   East  28.25             145        250.0   
3       P004  29.000000  Female   West  24.10             118        190.0   
4       P005  42.247368    Male  North  31.20             150        270.0   

   glucose  disease_risk  
0     95.0             0  
1    110.0             1  
2    140.0             1  
3     98.0             0  
4    160.0             1  

Missing Values
patient_id        0
age               0
gender            0
region            0
bmi               0
blood_pressure    0
cholesterol       0
glucose           0
disease_risk      0
dtype: int64

Dataset Shape
(200, 9)

Final Dataset Saved Successfully!


**Q7. Brief Report**

**1. Which imputation strategy was most effective?**

**Ans:** Median Imputation was the most effective for BMI because the dataset contained outliers. Most Frequent Imputation worked well for the categorical columns (Gender and Region). 

**2. Which outlier handling method preserved data quality best?**

**Ans:** Winsorization preserved data quality better because it capped extreme values instead of removing patient records, keeping the dataset size unchanged

**3. How did data cleaning improve dataset usability?**

**Ans:** Data cleaning removed missing values and handled outliers, resulting in a complete and consistent dataset. This improves the accuracy and reliability of data analysis and machine learning models.